# Phase 7B Capability Stress Suite Overview
This notebook outlines the design and implementation of a deterministic
validation layer for the Concierge multi‑agent system. The suite exercises
multi‑root orchestration, cross‑task synthesis, agent routing, critic
convergence, memory recall and structured output quality. It is intended
to be CI‑friendly and repeatable.

## 1. Import Required Modules
We'll import the necessary standard libraries and key classes from the
project, including the coordinator (`SacredTimeline`), concurrency manager,
memory store and agents for observation.  The configuration will be locked
for determinism.

In [ ]:
import os
import json
import logging
import asyncio

from sacred_timeline import SacredTimeline
from concurrency import AsyncConcurrencyManager
from memory.memory_store import MemoryStore
from agents.critic_agent import CriticAgent
from agents.synthesizer_agent import SynthesizerAgent

# a small helper from the earlier harness
def _load_backup_entries(path: str):
    entries = []
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as fh:
            for line in fh:
                try:
                    entries.append(json.loads(line))
                except Exception:
                    continue
    return entries

print("imports complete")

## 2. Define CAPABILITY_TESTS
The suite is driven by structured test cases. Each test includes an `id`,
a `goal`, a list of `expected_agents` and an optional
`requires_memory` flag.

For reproducibility we also embed a deterministic plan for each test.

In [ ]:
CAPABILITY_TESTS = [
    {
        "id": "framework_comparison",
        "goal": "Research two Python web frameworks, compare performance tradeoffs, implement minimal example snippets, and recommend one for high-scale APIs.",
        "expected_agents": ["ResearchAgent", "CodingAgent", "SynthesizerAgent", "CriticAgent"],
        "plan": [
            {"task_id": "t1", "title": "Framework Research", "instructions": "Research Python web frameworks and compare performance tradeoffs.", "depends_on": []},
            {"task_id": "t2", "title": "Example Code", "instructions": "Write a minimal example snippet for a web API in one of the frameworks.", "depends_on": []},
        ],
    },
    {
        "id": "data_pipeline_design",
        "goal": "Design a scalable data ingestion pipeline, provide architectural reasoning, and include a sample processing module.",
        "expected_agents": ["ResearchAgent", "CodingAgent", "SynthesizerAgent"],
        "plan": [
            {"task_id": "t1", "title": "Pipeline Research", "instructions": "Research scalable data ingestion patterns and architectures.", "depends_on": []},
            {"task_id": "t2", "title": "Sample Module", "instructions": "Provide sample code for a processing module in the pipeline.", "depends_on": []},
        ],
    },
    {
        "id": "conflicting_requirements",
        "goal": "Propose a system that is both fully decentralized and centrally controlled; explain tradeoffs and reconcile contradictions.",
        "expected_agents": ["ResearchAgent", "SynthesizerAgent", "CriticAgent"],
        "plan": [
            {"task_id": "t1", "title": "Decentralization vs Centralization", "instructions": "Research the tradeoffs between decentralization and central control.", "depends_on": []},
            {"task_id": "t2", "title": "Reconciliation", "instructions": "Describe how a system could reconcile these conflicting requirements.", "depends_on": []},
        ],
    },
    {
        "id": "memory_recall_test",
        "goal": "Based on previous framework research, summarize prior findings and refine the recommendation.",
        "expected_agents": ["SynthesizerAgent", "CriticAgent"],
        "requires_memory": True,
        "plan": [
            {"task_id": "t1", "title": "Memory Summary", "instructions": "Recall previous framework research from memory and summarize the key points.", "depends_on": []},
        ],
    },
]

print("tests defined")

## 3. Implement capability_tests.py
We'll expose the same `CAPABILITY_TESTS` list from a module so other
scripts (like the harness) can import it.

This file already exists in the repository.

In [ ]:
import capability_tests
print(capability_tests.CAPABILITY_TESTS[:1])

## 4. Implement validation helpers
A collection of small functions check the structure and expected
behaviour of each test result.  They enforce status, summary presence,
agent observations, depth/refine limits, concurrency bounds, critic
approval, and memory recall when requested.

In [ ]:
def _get_agent_types(entries):
    return list({e.get("metadata", {}).get("agent_type") for e in entries if e.get("metadata")})


def _compute_checks(test, result, agent_types, concurrency_peak):
    checks = {}
    checks["status_success"] = result.get("status") == "success"
    checks["final_summary_non_empty"] = bool(result.get("final", {}).get("summary"))
    checks["structured_key_points"] = bool(result.get("final", {}).get("structured", {}).get("key_points"))
    checks["synthesizer_ran"] = "SynthesizerAgent" in agent_types

    max_depth = 3
    max_refine = 2
    checks["depth_ok"] = all(t.get("depth", 0) <= max_depth for t in (result.get("task_map") or {}).values())
    checks["refine_ok"] = all(t.get("refine_count", 0) <= max_refine for t in (result.get("task_map") or {}).values())

    for ag in test.get("expected_agents", []):
        checks[f"agent_{ag}"] = ag in agent_types

    checks["concurrency_ok"] = concurrency_peak <= 1
    checks["critic_approved"] = result.get("evaluation", {}).get("decision") == "approve"

    if test.get("requires_memory"):
        mem_hit = any("context:" in (t.get("output") or "") for t in (result.get("task_map") or {}).values())
        checks["memory_hit"] = mem_hit
        checks["reflection_reused"] = mem_hit

    return checks

print("validation helpers ready")

## 5. Implement capability_harness.py
This script iterates through `CAPABILITY_TESTS`, runs the coordinator
with deterministic settings (max concurrent agents = 1, depth 3, refine 2),
and aggregates results.  It also logs an execution timeline and task
attributes for observability.

In [ ]:
import capability_harness  # verify import
print("harness imported")

## 6. Run Harness and Inspect Output
Execute the harness and examine its printed lines and JSON summary.  The
script will exit with non‑zero status if any test fails.

In [ ]:
import subprocess, sys

proc = subprocess.run([sys.executable, "capability_harness.py"], capture_output=True, text=True)
print(proc.stdout)
print(proc.stderr)
print(f"exit code: {proc.returncode}")

In [ ]:
import capability_harness

# run harness in-process to observe output directly
asyncio.run(capability_harness.main())
